# Student Depression Dataset - Exploratory Data Analysis

This notebook explores the **Student Depression Dataset** to understand the data distribution, identify patterns, and gain insights before model training.

**Target variable:** `Depression` (1 = Depressed, 0 = Not Depressed)

In [ ]:
# Cell 2 -- Import all libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)

print('Libraries loaded successfully!')

In [ ]:
# Cell 3 -- Load the CSV and inspect basic info
BASE_DIR  = os.path.dirname(os.path.abspath('__file__'))  # notebook dir
DATA_PATH = os.path.join(BASE_DIR, '..', 'Student Depression Dataset.csv')

df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape}')
print(f'\nFirst 5 rows:')
df.head()

In [ ]:
# Cell 4 -- df.info() and df.describe()
print('=== df.info() ===')
df.info()
print('\n=== df.describe() ===')
df.describe()

In [ ]:
# Cell 5 -- Check and print missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)
print(f'\nTotal missing: {missing.sum()}')

In [ ]:
# Cell 6 -- Countplot of the target variable
fig, ax = plt.subplots(figsize=(6, 4))
target_counts = df['Depression'].value_counts().rename({0: 'Not Depressed (0)', 1: 'Depressed (1)'})
sns.countplot(data=df, x='Depression', palette=['#2ecc71', '#e74c3c'], ax=ax)
ax.set_xticklabels(['Not Depressed (0)', 'Depressed (1)'])
ax.set_title('Class Distribution: Depression')
ax.set_xlabel('Depression Status')
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{p.get_height():,}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 9), textcoords='offset points')
plt.tight_layout()
plt.show()
print(f"\nDepression rate: {df['Depression'].mean()*100:.1f}%")

In [ ]:
# Cell 7 -- Heatmap of correlations between all numeric columns
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

fig, ax = plt.subplots(figsize=(14, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, linewidths=0.5,
    ax=ax, annot_kws={'size': 8}
)
ax.set_title('Correlation Heatmap -- Numeric Features')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 8 -- Histograms for all numeric columns
numeric_df = df.select_dtypes(include=[np.number])
n_cols = 3
n_rows = int(np.ceil(len(numeric_df.columns) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(numeric_df.columns):
    axes[i].hist(numeric_df[col].dropna(), bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

# Hide any unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Histograms -- All Numeric Columns', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 -- Bar charts: average depression rate by Gender, Sleep Duration, Dietary Habits
CAT_COLS = ['Gender', 'Sleep Duration', 'Dietary Habits']

fig, axes = plt.subplots(1, len(CAT_COLS), figsize=(16, 5))

for ax, col in zip(axes, CAT_COLS):
    if col not in df.columns:
        continue
    avg_dep = df.groupby(col)['Depression'].mean().sort_values(ascending=False)
    bars = ax.bar(avg_dep.index, avg_dep.values * 100,
                  color=plt.cm.Blues(np.linspace(0.4, 0.9, len(avg_dep))))
    ax.set_title(f'Avg. Depression Rate by {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Depression Rate (%)')
    ax.set_ylim(0, 100)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, avg_dep.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{val*100:.1f}%', ha='center', va='bottom', fontsize=9)

plt.suptitle('Depression Rate by Categorical Features', fontsize=14)
plt.tight_layout()
plt.show()

## Summary of EDA Findings

### Key Observations

1. **Dataset Size**: The dataset contains **27,901 rows** and **18 columns**.

2. **Class Balance**: The dataset shows a moderate imbalance:
   - Approximately 58% of students are classified as **Depressed (1)**
   - Approximately 42% are classified as **Not Depressed (0)**

3. **Missing Values**: Only the `Financial Stress` column has a small number of missing values, which are easily handled by median imputation.

4. **Top Correlating Features with Depression**:
   - `Have you ever had suicidal thoughts?` — strong positive correlation
   - `Academic Pressure` — positive correlation
   - `Financial Stress` — positive correlation
   - `Study Satisfaction` — negative correlation (higher satisfaction -> lower depression)

5. **Sleep Duration**: Students sleeping "Less than 5 hours" have a notably higher depression rate compared to those sleeping "7-8 hours".

6. **Dietary Habits**: Students with "Unhealthy" dietary habits show a higher average depression rate than those with "Healthy" habits.

7. **Gender**: Slight difference in depression rates between Male and Female students.

8. **CGPA**: A bimodal distribution is visible, with many students clustered at both lower and higher CGPA ranges.

### Preprocessing Recommendations
- Encode categorical columns (Gender, Sleep Duration, Dietary Habits, etc.) using LabelEncoder
- Scale all numeric features with StandardScaler before training
- Engineer `Stress_Score`, `Sleep_Category`, and `Study_Work_Balance` as additional features